# مسئلهٔ ۱ — V2 / مدل A1: ResNet18 + Mean Temporal Pooling

هر نمونه یک sequence کامل با شکل `[16, 3, 224, 320]` و **یک** برچسب ویدئویی دارد. ResNet18 برای هر فریم feature 512بعدی می‌سازد؛ سپس میانگین زمانی featureها به یک logit تصادف تبدیل می‌شود.

برای baseline اول، encoder ازپیش‌آموزش‌دیدهٔ ImageNet freeze است و فقط head آموزش می‌بیند. این تصمیم عمدی است: دادهٔ ما کوچک و اجرای محلی CPU است. در مرحلهٔ بعد، همین baseline با fine-tuning محدود و سپس attention مقایسه خواهد شد.

معیارها در این نوت‌بوک **clip-level validation** هستند؛ برای inference نهاییِ MP4 کامل بعداً sliding-window را جداگانه ارزیابی می‌کنیم.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import random

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
import torch
from tqdm.auto import tqdm

DATA_ROOT = Path(r'P:\NexarCollisionData')
SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
FRAME_INDEX_PATH = DATA_ROOT / 'frame_cache_index_v2.csv'
FEATURE_CACHE_PATH = DATA_ROOT / 'processed_v2' / 'resnet18_imagenet_features_v2_w2_16x224x320.pt'
MODEL_DIR = DATA_ROOT / 'models_v2'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'resnet18_mean_pooling_frozen'
NUM_FRAMES = 16
FEATURE_DIM = 512
FRAME_BATCH_SIZE = 2      # sequences; the encoder sees 2 × 16 = 32 frames at once
HEAD_BATCH_SIZE = 64
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
assert SEQUENCE_MANIFEST_PATH.exists(), 'Run notebook 08 first.'
assert FRAME_INDEX_PATH.exists(), 'Run notebook 09 first.'

Device: cpu


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

sequence_manifest = pd.read_csv(SEQUENCE_MANIFEST_PATH).copy()
frame_index = pd.read_csv(FRAME_INDEX_PATH).copy()
sequence_manifest['video_id'] = sequence_manifest['video_id'].astype(str)
sequence_manifest['label'] = sequence_manifest['label'].astype(int)
frame_index['video_id'] = frame_index['video_id'].astype(str)
frame_index['label'] = frame_index['label'].astype(int)
frame_index['frame_valid'] = frame_index['frame_valid'].astype(str).str.lower().eq('true')

assert len(sequence_manifest) == 600
assert sequence_manifest['sequence_id'].is_unique
assert len(frame_index) == 600 * NUM_FRAMES
assert frame_index['frame_valid'].all(), 'All cached frames must be valid before training.'
assert frame_index['frame_path'].map(lambda path: Path(path).is_file()).all(), 'A cached frame is missing.'
assert frame_index.groupby('sequence_id').size().eq(NUM_FRAMES).all()

sequence_manifest = sequence_manifest.sort_values('video_id', key=lambda series: series.astype(int)).reset_index(drop=True)
frame_groups = {
    sequence_id: group.sort_values('frame_index').reset_index(drop=True)
    for sequence_id, group in frame_index.groupby('sequence_id', sort=False)
}
assert set(sequence_manifest['sequence_id']) == set(frame_groups)

print('Frozen V2 split:')
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))

Frozen V2 split:


label,0,1
split,,
train,240,240
validation,60,60


In [3]:
class SequenceFrameDataset(Dataset):
    """Returns one ordered 16-frame tensor and one video-level label per item."""
    def __init__(self, sequence_table: pd.DataFrame, grouped_frames: dict[str, pd.DataFrame]):
        self.sequence_table = sequence_table.reset_index(drop=True)
        self.grouped_frames = grouped_frames

    def __len__(self) -> int:
        return len(self.sequence_table)

    @staticmethod
    def load_and_normalize_rgb(path: str) -> torch.Tensor:
        frame_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if frame_bgr is None:
            raise RuntimeError(f'Cannot read cached frame: {path}')
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(frame_rgb.copy()).permute(2, 0, 1).float().div_(255.0)
        return (tensor - IMAGENET_MEAN) / IMAGENET_STD

    def __getitem__(self, index: int):
        sequence_row = self.sequence_table.iloc[index]
        frames = self.grouped_frames[sequence_row.sequence_id]
        assert len(frames) == NUM_FRAMES
        images = torch.stack([self.load_and_normalize_rgb(path) for path in frames['frame_path']])
        return images, int(sequence_row.label), sequence_row.sequence_id, sequence_row.video_id

all_frame_dataset = SequenceFrameDataset(sequence_manifest, frame_groups)
example_images, example_label, example_sequence_id, example_video_id = all_frame_dataset[0]
assert example_images.shape == (NUM_FRAMES, 3, 224, 320)
print({
    'images_shape': tuple(example_images.shape),
    'label': example_label,
    'sequence_id': example_sequence_id,
    'video_id': example_video_id,
})

{'images_shape': (16, 3, 224, 320), 'label': 1, 'sequence_id': 'V2-W2_0', 'video_id': '0'}


## استخراج featureهای ثابت ResNet18

این کار فقط یک بار انجام می‌شود و featureهای ImageNet را cache می‌کند. اجرای مجدد، در صورت سازگاری cache با همین ۶۰۰ sequence، featureها را دوباره محاسبه نمی‌کند. در این baseline augmentation نداریم، چون featureها ثابت‌اند؛ augmentation سازگار در سطح کل sequence در نسخهٔ fine-tuned اضافه می‌شود.

In [4]:
expected_sequence_ids = sequence_manifest['sequence_id'].tolist()
reuse_feature_cache = False
feature_payload = None

if FEATURE_CACHE_PATH.exists():
    cached_payload = torch.load(FEATURE_CACHE_PATH, map_location='cpu', weights_only=False)
    if (cached_payload.get('sequence_ids') == expected_sequence_ids
            and tuple(cached_payload.get('features', torch.empty(0)).shape) == (600, NUM_FRAMES, FEATURE_DIM)
            and cached_payload.get('preprocessing_version') == 'v2_w2_rgb_letterbox_replicate_224x320'):
        feature_payload = cached_payload
        reuse_feature_cache = True

if not reuse_feature_cache:
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    backbone = models.resnet18(weights=weights)
    encoder = nn.Sequential(*list(backbone.children())[:-1]).to(device).eval()
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)

    loader = DataLoader(
        all_frame_dataset, batch_size=FRAME_BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=(device.type == 'cuda'),
    )
    feature_batches, label_batches, sequence_ids, video_ids = [], [], [], []
    with torch.inference_mode():
        for images, labels, batch_sequence_ids, batch_video_ids in tqdm(loader, desc='Extracting ResNet18 features'):
            batch_size, time_steps, channels, height, width = images.shape
            flattened_images = images.reshape(batch_size * time_steps, channels, height, width).to(device)
            features = encoder(flattened_images).flatten(1).reshape(batch_size, time_steps, FEATURE_DIM).cpu()
            feature_batches.append(features)
            label_batches.append(labels.cpu())
            sequence_ids.extend(batch_sequence_ids)
            video_ids.extend(batch_video_ids)

    feature_payload = {
        'features': torch.cat(feature_batches),
        'labels': torch.cat(label_batches).long(),
        'sequence_ids': sequence_ids,
        'video_ids': video_ids,
        'splits': sequence_manifest['split'].tolist(),
        'preprocessing_version': 'v2_w2_rgb_letterbox_replicate_224x320',
        'encoder': 'ResNet18_Weights.IMAGENET1K_V1 (frozen)',
    }
    torch.save(feature_payload, FEATURE_CACHE_PATH)

features = feature_payload['features'].float()
labels = feature_payload['labels'].long()
assert features.shape == (600, NUM_FRAMES, FEATURE_DIM)
assert feature_payload['sequence_ids'] == expected_sequence_ids
assert labels.tolist() == sequence_manifest['label'].tolist()
print({
    'feature_cache': str(FEATURE_CACHE_PATH),
    'reused_existing_cache': reuse_feature_cache,
    'feature_shape': tuple(features.shape),
})

Extracting ResNet18 features: 100%|██████████| 300/300 [11:47<00:00,  2.36s/it]

{'feature_cache': 'P:\\NexarCollisionData\\processed_v2\\resnet18_imagenet_features_v2_w2_16x224x320.pt', 'reused_existing_cache': False, 'feature_shape': (600, 16, 512)}


In [5]:
class SequenceFeatureDataset(Dataset):
    def __init__(self, features: torch.Tensor, labels: torch.Tensor, indices: np.ndarray):
        self.features = features
        self.labels = labels
        self.indices = torch.as_tensor(indices, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int):
        source_index = self.indices[index]
        return self.features[source_index], self.labels[source_index], int(source_index)

class ResNet18MeanPoolingHead(nn.Module):
    def __init__(self, feature_dim: int = FEATURE_DIM, dropout: float = 0.30):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 1),
        )

    def forward(self, sequence_features: torch.Tensor, frame_mask: torch.Tensor | None = None) -> torch.Tensor:
        if frame_mask is None:
            video_features = sequence_features.mean(dim=1)
        else:
            weights = frame_mask.float().unsqueeze(-1)
            video_features = (sequence_features * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)
        return self.classifier(video_features).squeeze(1)

train_indices = np.flatnonzero(sequence_manifest['split'].eq('train').to_numpy())
validation_indices = np.flatnonzero(sequence_manifest['split'].eq('validation').to_numpy())
assert len(train_indices) == 480 and len(validation_indices) == 120

train_loader = DataLoader(SequenceFeatureDataset(features, labels, train_indices), batch_size=HEAD_BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(SequenceFeatureDataset(features, labels, validation_indices), batch_size=HEAD_BATCH_SIZE, shuffle=False, num_workers=0)

model = ResNet18MeanPoolingHead().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
print(model)

ResNet18MeanPoolingHead(
  (classifier): Sequential(
    (0): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=512, out_features=1, bias=True)
  )
)


In [6]:
def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'confusion_matrix': confusion_matrix(y_true, predictions).tolist(),
    }

def evaluate(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_labels, all_probabilities, all_indices = [], [], []
    with torch.inference_mode():
        for batch_features, batch_labels, batch_indices in loader:
            logits = model(batch_features.to(device))
            all_labels.append(batch_labels.numpy())
            all_probabilities.append(torch.sigmoid(logits).cpu().numpy())
            all_indices.append(batch_indices.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probabilities), np.concatenate(all_indices)

best_pr_auc = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []
best_model_path = MODEL_DIR / f'{MODEL_NAME}_best.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for batch_features, batch_labels, _ in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.float().to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(batch_features), batch_labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(batch_labels)

    validation_labels, validation_probabilities, _ = evaluate(model, validation_loader)
    metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, threshold=0.5)
    record = {
        'epoch': epoch,
        'train_loss': running_loss / len(train_loader.dataset),
        'validation_accuracy_at_0_5': metrics_at_05['accuracy'],
        'validation_f1_at_0_5': metrics_at_05['f1'],
        'validation_recall_at_0_5': metrics_at_05['recall'],
        'validation_pr_auc': metrics_at_05['pr_auc'],
        'validation_roc_auc': metrics_at_05['roc_auc'],
    }
    history.append(record)
    print(record)

    if record['validation_pr_auc'] > best_pr_auc:
        best_pr_auc = record['validation_pr_auc']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'validation_pr_auc': best_pr_auc,
            'model_name': MODEL_NAME,
            'feature_dim': FEATURE_DIM,
            'num_frames': NUM_FRAMES,
            'encoder': 'ResNet18_Weights.IMAGENET1K_V1 (frozen)',
        }, best_model_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}; best epoch: {best_epoch}')
            break

history_df = pd.DataFrame(history)
history_path = MODEL_DIR / f'{MODEL_NAME}_training_history.csv'
history_df.to_csv(history_path, index=False)
print(f'Best epoch by validation PR-AUC: {best_epoch}, PR-AUC={best_pr_auc:.4f}')

{'epoch': 1, 'train_loss': 0.6573042074839274, 'validation_accuracy_at_0_5': 0.5666666666666667, 'validation_f1_at_0_5': 0.3953488372093023, 'validation_recall_at_0_5': 0.2833333333333333, 'validation_pr_auc': 0.676049689634811, 'validation_roc_auc': 0.7013888888888888}
{'epoch': 2, 'train_loss': 0.6257477601369222, 'validation_accuracy_at_0_5': 0.675, 'validation_f1_at_0_5': 0.6829268292682927, 'validation_recall_at_0_5': 0.7, 'validation_pr_auc': 0.7004856257849761, 'validation_roc_auc': 0.7336111111111112}
{'epoch': 3, 'train_loss': 0.6169376929601034, 'validation_accuracy_at_0_5': 0.7, 'validation_f1_at_0_5': 0.7, 'validation_recall_at_0_5': 0.7, 'validation_pr_auc': 0.709795033260029, 'validation_roc_auc': 0.7408333333333333}
{'epoch': 4, 'train_loss': 0.5899250229199727, 'validation_accuracy_at_0_5': 0.6916666666666667, 'validation_f1_at_0_5': 0.6782608695652174, 'validation_recall_at_0_5': 0.65, 'validation_pr_auc': 0.7105179368859025, 'validation_roc_auc': 0.7472222222222222}
{

In [7]:
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
validation_labels, validation_probabilities, validation_source_indices = evaluate(model, validation_loader)

threshold_rows = []
for threshold in np.round(np.arange(0.10, 0.901, 0.01), 2):
    threshold_rows.append(binary_metrics(validation_labels, validation_probabilities, float(threshold)))
threshold_table = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_table.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
selected_threshold = float(best_threshold_row['threshold'])
metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, threshold=0.5)
metrics_at_selected_threshold = binary_metrics(validation_labels, validation_probabilities, threshold=selected_threshold)

validation_rows = sequence_manifest.iloc[validation_source_indices].copy().reset_index(drop=True)
validation_rows['positive_probability'] = validation_probabilities
validation_rows['prediction_at_0_5'] = (validation_probabilities >= 0.5).astype(int)
validation_rows['prediction_at_selected_threshold'] = (validation_probabilities >= selected_threshold).astype(int)
validation_rows['selected_threshold'] = selected_threshold

predictions_path = MODEL_DIR / f'{MODEL_NAME}_validation_predictions.csv'
threshold_path = MODEL_DIR / f'{MODEL_NAME}_threshold_curve.csv'
metrics_path = MODEL_DIR / f'{MODEL_NAME}_metrics.json'
validation_rows.to_csv(predictions_path, index=False)
threshold_table.to_csv(threshold_path, index=False)

metrics_payload = {
    'model_name': MODEL_NAME,
    'evaluation_scope': 'clip-level validation on the fixed V2-W2 sequences; not full-MP4 sliding-window inference',
    'selection_metric': 'validation PR-AUC at the saved checkpoint',
    'best_epoch': int(checkpoint['epoch']),
    'selected_threshold_by_validation_f1': selected_threshold,
    'metrics_at_threshold_0_5': metrics_at_05,
    'metrics_at_selected_threshold': metrics_at_selected_threshold,
    'encoder': 'ResNet18 ImageNet frozen',
    'temporal_aggregator': 'mean pooling over 16 frame features',
    'input_shape_per_sequence': [NUM_FRAMES, 3, 224, 320],
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= 0.5).astype(int), ax=axes[0], colorbar=False)
axes[0].set_title('Validation, threshold = 0.50')
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= selected_threshold).astype(int), ax=axes[1], colorbar=False)
axes[1].set_title(f'Validation, threshold = {selected_threshold:.2f}')
figure.tight_layout()
confusion_path = MODEL_DIR / f'{MODEL_NAME}_confusion_matrices.png'
figure.savefig(confusion_path, dpi=160)
plt.close(figure)

print('Metrics at threshold 0.50:')
print(metrics_at_05)
print('Metrics at the validation-selected threshold:')
print(metrics_at_selected_threshold)
print(f'Model: {best_model_path}')
print(f'Predictions: {predictions_path}')
print(f'Metrics: {metrics_path}')

Metrics at threshold 0.50:
{'threshold': 0.5, 'accuracy': 0.7666666666666667, 'precision': 0.7962962962962963, 'recall': 0.7166666666666667, 'f1': 0.7543859649122807, 'roc_auc': 0.7675000000000001, 'pr_auc': 0.7275106287354545, 'confusion_matrix': [[49, 11], [17, 43]]}
Metrics at the validation-selected threshold:
{'threshold': 0.47, 'accuracy': 0.7666666666666667, 'precision': 0.7758620689655172, 'recall': 0.75, 'f1': 0.7627118644067796, 'roc_auc': 0.7675000000000001, 'pr_auc': 0.7275106287354545, 'confusion_matrix': [[47, 13], [15, 45]]}
Model: P:\NexarCollisionData\models_v2\resnet18_mean_pooling_frozen_best.pt
Predictions: P:\NexarCollisionData\models_v2\resnet18_mean_pooling_frozen_validation_predictions.csv
Metrics: P:\NexarCollisionData\models_v2\resnet18_mean_pooling_frozen_metrics.json


## تفسیر درست خروجی

- معیار اصلی: F1 کلاس تصادف در آستانهٔ انتخاب‌شده روی validation.
- قید ایمنی: Recall تصادف را کنار F1 گزارش می‌کنیم؛ Accuracy به‌تنهایی معیار انتخاب نیست.
- PR-AUC معیار مستقل از آستانه برای انتخاب checkpoint است.
- این مدل baseline A1 است. اگر از V1 بهتر شد، گام بعد mean+max/attention و سپس fine-tuning محدود ResNet18 خواهد بود.
- چون positive window در آموزش با `time_of_event` ساخته شده، این عدد هنوز معیار نهاییِ «هر MP4 کامل» نیست؛ برای خروجی نهایی باید sliding-window inference را روی validation پیاده‌سازی و ارزیابی کنیم.